In [7]:
import pandas as pd

# Load all tables from your MySQL database into pandas DataFrames
movies_df = pd.read_sql("SELECT * FROM movies", engine)
genres_df = pd.read_sql("SELECT * FROM genres", engine)
movie_genres_df = pd.read_sql("SELECT * FROM movie_genres", engine)
cast_df = pd.read_sql("SELECT * FROM cast", engine)
crew_df = pd.read_sql("SELECT * FROM crew", engine)
movie_keywords_df = pd.read_sql("SELECT * FROM movie_keywords", engine)

#1. List the total number of unique movies, genres, cast members and crew members collected across all tables.
unique_movies = movies_df['movie_id'].nunique()
unique_genres = genres_df['genre_id'].nunique()
unique_cast = cast_df['person_id'].nunique()
unique_crew = crew_df['person_id'].nunique()

summary_counts = pd.DataFrame({
    'Entity': ['Movies', 'Genres', 'Cast Members', 'Crew Members'],
    'Total Unique Count': [
        unique_movies,
        unique_genres,
        unique_cast,
        unique_crew,
    ],
})
display(summary_counts)

,Entity,Total Unique Count
0,Movies,2503
1,Genres,19
2,Cast Members,11300
3,Crew Members,2848


In [9]:
#2. List all movies with missing or zero budget/revenue values, and decide how these should be treated for the rest of the analysis.

missing_financials = movies_df[
    (movies_df['budget'] == 0)
    | (movies_df['revenue'] == 0)
    | movies_df['budget'].isna()
    | movies_df['revenue'].isna()
]
print(f'Total movies with zero or missing budget/revenue: {len(missing_financials)}')
display(missing_financials[['title', 'release_date', 'budget', 'revenue']].head(10))

Total movies with zero or missing budget/revenue: 133


,title,release_date,budget,revenue
324,Zack Snyder's Justice League,2021-03-18,70000000.0,0.0
340,Bird Box,2018-12-13,19800000.0,0.0
403,365 Days,2020-02-07,0.0,9458590.0
468,To All the Boys I've Loved Before,2018-08-17,0.0,0.0
554,Prey,2022-08-02,65000000.0,0.0
598,The Kissing Booth,2018-05-11,0.0,0.0
717,Enola Holmes,2020-09-22,21000000.0,0.0
723,Extraction,2020-04-23,65000000.0,0.0
754,"Love, Rosie",2014-10-16,0.0,4439431.0
842,Bright,2017-12-22,90000000.0,0.0


In [10]:
#3. List the profit (revenue − budget) and return on investment for every movie that has valid budget and revenue figures.

# Filter for valid, non-zero figures
valid_fin_df = movies_df[
    (movies_df['budget'] > 0) & (movies_df['revenue'] > 0)
].copy()

valid_fin_df['profit'] = valid_fin_df['revenue'] - valid_fin_df['budget']
valid_fin_df['roi'] = valid_fin_df['profit'] / valid_fin_df['budget']

display(
    valid_fin_df[['title', 'budget', 'revenue', 'profit', 'roi']].head(10)
)

,title,budget,revenue,profit,roi
0,Interstellar,165000000.0,7.466067e+08,5.816067e+08,3.524889
1,Inception,160000000.0,8.390306e+08,6.790306e+08,4.243941
2,The Avengers,220000000.0,1.518816e+09,1.298816e+09,5.903707
3,The Dark Knight,185000000.0,1.004558e+09,8.195584e+08,4.430046
4,Avatar,237000000.0,2.923706e+09,2.686706e+09,11.336312
5,Deadpool,58000000.0,7.828373e+08,7.248373e+08,12.497196
6,Fight Club,63000000.0,1.008538e+08,3.785375e+07,0.600853
7,Avengers: Infinity War,300000000.0,2.052415e+09,1.752415e+09,5.841383
8,The Shawshank Redemption,25000000.0,2.834147e+07,3.341469e+06,0.133659
9,Pulp Fiction,8000000.0,2.139288e+08,2.059288e+08,25.741095


In [11]:
#4. List the top 15 movies by return on investment, considering only movies with a budget of at least $1 million.

filtered_roi = valid_fin_df[valid_fin_df['budget'] >= 1_000_000]
top_15_roi = filtered_roi.sort_values(by='roi', ascending=False).head(15)

display(
    top_15_roi[['title', 'budget', 'revenue', 'profit', 'roi']]
)

,title,budget,revenue,profit,roi
1767,Dragon Ball Super: Broly,1000000.0,125002821.0,124002821.0,124.002821
535,Snow White and the Seven Dwarfs,1488423.0,184925486.0,183437063.0,123.242561
1791,The Rocky Horror Picture Show,1400000.0,171181400.0,169781400.0,121.272429
467,Rocky,1000000.0,117253345.0,116253345.0,116.253345
1227,Gone with the Wind,4000000.0,402352579.0,398352579.0,99.588145
716,The Jungle Book,4000000.0,378000000.0,374000000.0,93.500000
635,Cinderella,2900000.0,263600000.0,260700000.0,89.896552
372,Saw,1200000.0,104045735.0,102845735.0,85.704779
719,One Hundred and One Dalmatians,3600000.0,303000000.0,299400000.0,83.166667
263,E.T. the Extra-Terrestrial,10500000.0,797307407.0,786807407.0,74.934039


In [12]:
#5. List the number of movies released per year, and identify which year had the highest movie output in the dataset.

movies_df['release_year'] = pd.to_datetime(
    movies_df['release_date'], errors='coerce'
).dt.year
movies_per_year = (
    movies_df['release_year']
    .value_counts()
    .reset_index(name='movie_count')
    .sort_values(by='release_year')
)

peak_year = movies_per_year.loc[movies_per_year['movie_count'].idxmax()]
print(
    f'Year with highest output: {int(peak_year["release_year"])} with'
    f' {int(peak_year["movie_count"])} movies.'
)
display(movies_per_year.tail(10))

Year with highest output: 2016 with 121 movies.


,release_year,movie_count
2,2017,109
4,2018,102
5,2019,100
21,2020,52
14,2021,67
15,2022,65
18,2023,62
24,2024,43
29,2025,26
45,2026,8


In [13]:
#6. List all movies whose runtime is a statistical outlier compared to the rest of the dataset.

runtimes = movies_df['runtime'].dropna()
Q1 = runtimes.quantile(0.25)
Q3 = runtimes.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_runtimes = movies_df[
    (movies_df['runtime'] < lower_bound)
    | (movies_df['runtime'] > upper_bound)
]
print(
    f'Outlier thresholds — Lower: {lower_bound:.1f} mins, Upper:'
    f' {upper_bound:.1f} mins'
)
display(
    outlier_runtimes[['title', 'runtime', 'release_date']].sort_values(
        by='runtime', ascending=False
    )
)

Outlier thresholds — Lower: 58.5 mins, Upper: 166.5 mins


,title,runtime,release_date
324,Zack Snyder's Justice League,242,2021-03-18
1227,Gone with the Wind,238,1939-12-15
829,Once Upon a Time in America,229,1984-05-23
1610,Lawrence of Arabia,228,1962-12-11
1774,Ben-Hur,212,1959-11-18
587,The Irishman,209,2019-11-01
1277,Seven Samurai,207,1954-04-26
1304,Killers of the Flower Moon,206,2023-10-18
187,The Godfather Part II,202,1974-12-20
20,The Lord of the Rings: The Return of the King,201,2003-12-17


In [14]:
#7. List each movie's primary genre (the first genre listed) and compare average popularity across primary genres.

# Merge movies with movie_genres and genres to isolate the first assigned genre per movie
merged_genres = movies_df.merge(movie_genres_df, on='movie_id').merge(
    genres_df, on='genre_id'
)

# Keep the first listed genre per movie_id
primary_genre_df = merged_genres.drop_duplicates(subset=['movie_id'], keep='first')

popularity_by_genre = (
    primary_genre_df.groupby('genre_name')['popularity']
    .mean()
    .reset_index(name='avg_popularity')
    .sort_values(by='avg_popularity', ascending=False)
)
display(popularity_by_genre)

,genre_name,avg_popularity
14,Science Fiction,21.261836
11,Music,18.324340
2,Animation,16.811212
7,Family,16.710635
0,Action,16.385188
16,War,15.470317
10,Horror,14.851835
1,Adventure,14.785510
8,Fantasy,13.022920
12,Mystery,12.427131


In [15]:
#8. List the top 10 most prolific actors by number of movies, along with their average movie rating across those movies.

actor_merged = cast_df.merge(
    movies_df[['movie_id', 'vote_average']], on='movie_id'
)

actor_stats = (
    actor_merged.groupby('actor_name')
    .agg(movie_count=('movie_id', 'nunique'), avg_rating=('vote_average', 'mean'))
    .reset_index()
)

top_10_actors = actor_stats.sort_values(by='movie_count', ascending=False).head(10)
display(top_10_actors)

,actor_name,movie_count,avg_rating
9526,Samuel L. Jackson,39,6.938692
9102,Robert De Niro,38,7.228316
1255,Brad Pitt,38,7.257289
5317,Johnny Depp,37,6.856973
10599,Tom Hanks,33,7.289758
11031,Willem Dafoe,32,7.114781
7090,Mark Wahlberg,32,6.630063
9630,Scarlett Johansson,32,7.056344
10586,Tom Cruise,31,7.040613
7814,Morgan Freeman,31,6.963742


In [16]:
#9. List any movie titles that appear more than once in the dataset, along with their release years, to check for duplicates or remakes.

duplicates = movies_df[
    movies_df.duplicated(subset=['title'], keep=False)
].sort_values(by='title')
display(
    duplicates[['title', 'release_date', 'budget', 'revenue', 'vote_average']]
)

,title,release_date,budget,revenue,vote_average
878,A Nightmare on Elm Street,1984-11-09,1.800000e+06,5.700451e+07,7.350
1928,A Nightmare on Elm Street,2010-04-30,3.500000e+07,1.156654e+08,5.525
261,Aladdin,1992-11-25,2.800000e+07,5.040502e+08,7.657
339,Aladdin,2019-05-22,1.830000e+08,1.054304e+09,7.081
166,Alice in Wonderland,2010-03-03,2.000000e+08,1.025467e+09,6.645
...,...,...,...,...,...
1737,The Thing,2011-10-12,3.500000e+07,3.150529e+07,6.253
744,Total Recall,1990-06-01,6.500000e+07,2.613179e+08,7.324
860,Total Recall,2012-08-02,1.250000e+08,1.988000e+08,6.033
252,Transformers,2007-06-27,1.500000e+08,7.097098e+08,6.843


In [17]:
#10. List movies whose popularity score and vote count don't follow the general pattern between the two — i.e., unusually high popularity with a low vote count, or vice versa.

# Use standard deviation residuals or simple ratio thresholds from regression trends
movies_df['pop_vote_ratio'] = movies_df['popularity'] / (
    movies_df['vote_count'] + 1
)

# Filtering anomalies: high popularity despite low vote counts (e.g. recent viral/hyped trailers or releases)
unusual_movies = movies_df[
    (movies_df['popularity'] > movies_df['popularity'].quantile(0.90))
    & (movies_df['vote_count'] < movies_df['vote_count'].quantile(0.50))
]

display(
    unusual_movies[
        ['title', 'popularity', 'vote_count', 'vote_average', 'release_date']
    ].sort_values(by='popularity', ascending=False)
)

,title,popularity,vote_count,vote_average,release_date
2388,Disclosure Day,394.3346,2328,7.427,2026-06-10
1315,Obsession,364.8722,4203,8.248,2026-05-13
2248,Backrooms,258.4743,2483,7.100,2026-05-27
1465,Michael,108.7248,3846,8.696,2026-04-22
2452,Mortal Kombat II,99.5231,2260,7.906,2026-05-06
1707,The Super Mario Galaxy Movie,97.1210,3343,8.261,2026-04-01
1378,Avatar: Fire and Ash,95.5319,4032,7.636,2025-12-17
2295,Lee Cronin's The Mummy,79.4601,2430,7.943,2026-04-15
1776,Zootopia 2,63.1069,3195,7.680,2025-11-26
1754,Captain America: Brave New World,60.3616,3233,6.000,2025-02-12
